# Generación de SBOMs

## Objetivo

Este notebook explica el proceso para generar **Software Bill of Materials (SBOMs)** para un conjunto de repositorios de código y cómo analizar los resultados de manera centralizada. Utilizaremos la herramienta **Syft** para la generación y la librería **Pandas** en Python para el análisis.

El flujo de trabajo completo está diseñado para ser reproducible y se basa en las siguientes etapas:

1.  **Descubrimiento de Repositorios**: Identificar los proyectos a los que se les generará un SBOM.
2.  **Generación de SBOMs**: Ejecutar Syft para analizar cada repositorio y crear un SBOM en formato JSON.
3.  **Análisis de Resultados**: Cargar todos los SBOMs generados en un DataFrame de Pandas para su inspección y análisis.


## Estructura de Directorios

El proyecto sigue una estructura organizada para separar los datos de entrada, los scripts y los resultados:

```
produccion_software_emi305/
├── data/
│   ├── repos/      # Directorio para los repositorios a analizar
│   └── results/    # Directorio para los SBOMs generados en JSON
├── nbs/            # Notebooks de Jupyter
└── scripts/        # Scripts de automatización
    └── generate_sboms.py
```

## Paso 1: Generación de SBOMs

El primer paso es ejecutar el script `generate_sboms.py`, que se encarga de orquestar la generación de los SBOMs.

Este script realiza las siguientes acciones:

1.  **Busca repositorios**: Escanea el directorio `data/repos` en busca de subdirectorios que contengan código fuente.
2.  **Ejecuta Syft**: Para cada repositorio encontrado, invoca a `syft` para analizar su contenido.
3.  **Guarda los resultados**: El SBOM generado para cada repositorio se guarda como un archivo JSON en el directorio `data/results`.

In [ ]:
#| eval: false
!python "../../scripts/generate_sboms.py"

## Paso 2: Análisis de los SBOMs con Pandas

Una vez que los SBOMs han sido generados, podemos cargarlos en un DataFrame de Pandas para analizarlos. Esto nos permite tener una visión consolidada de todas las dependencias y artefactos encontrados en los diferentes repositorios.

El siguiente código se encarga de:

1.  **Localizar los archivos JSON**: Busca todos los archivos `.json` en el directorio `data/results`.
2.  **Cargar y normalizar los datos**: Para cada archivo JSON, lo carga y utiliza `pd.json_normalize` para aplanar la estructura anidada de los artefactos.
3.  **Añadir información del repositorio**: Agrega una columna `repo` para identificar a qué repositorio pertenece cada artefacto.
4.  **Concatenar los resultados**: Une todos los DataFrames individuales en uno solo.

In [ ]:
import pandas as pd
from pathlib import Path
import json

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
path = project_root / 'data' / 'results'

df_all = pd.concat(
    [
        pd.json_normalize(json.load(open(file))["artifacts"]).assign(repo=file.stem)
        for file in path.glob("*-sbom.json")
    ],
    ignore_index=True
)

df_all.head()

## Exploración de los Resultados

Con los datos en un DataFrame, podemos realizar diversas consultas y análisis. Por ejemplo, podemos ver cuántos artefactos se encontraron por cada repositorio.

In [ ]:
df_all['repo'].value_counts()

También podemos ver las licencias más comunes encontradas en todas las dependencias.

In [ ]:
df_all['licenses'].explode().value_counts().head(10)

## Cómo Añadir Nuevos Repositorios al Análisis

El proyecto está diseñado para que sea sencillo añadir nuevos repositorios al proceso de generación de SBOMs. El sistema utiliza clones locales declarados en `data/repos.json` para gestionar los repositorios externos.

#### 1. Edita el archivo `data/repos.json`

Abre `data/repos.json` y agrega nuevos repositorios con esta estructura:

```json
{
  "url": "URL_DEL_REPOSITORIO_EN_GITHUB.git",
  "path": "data/repos/NOMBRE_DEL_REPOSITORIO"
}
```

- `url`: La URL `.git` del repositorio
- `path`: Ruta local donde se clonará (usa `data/repos/{nombre}`)

#### 2. Ejecuta en la terminal (esto clona los repositorios y los agrega como submódulos):

```bash
uv run python scripts/add_submodules.py
```

#### 3. Ejecuta la generación de SBOMs nuevamente:

```bash
uv run scripts/generate_sboms.py
```

O desde el notebook:
- Reinicia el kernel
- Ejecuta la celda con `!python ../../scripts/generate_sboms.py`
